In [1]:
from src import WhatsappClient

In [2]:
client = WhatsappClient()
await client.initialize_playwright()

2024-10-17 19:13:03,803 - INFO - Logger initialized.
2024-10-17 19:13:03,803 - INFO - Starting Playwright...
2024-10-17 19:13:03,987 - INFO - Launching chromium with persistent context...
2024-10-17 19:13:04,769 - INFO - chromium launched successfully.


In [3]:
await client.login()

2024-10-17 19:13:08,623 - INFO - User is already logged in. Skipping login step.
2024-10-17 19:13:09,288 - INFO - Waiting for WhatsApp chats to load...
2024-10-17 19:13:15,051 - INFO - WhatsApp chats loaded.


In [ ]:
# await client.search_pane_scroll_down()
await client.chat_pane_scroll_up()

In [ ]:
await client.open_chat_panel("Prathwik")

In [ ]:
from tabulate import tabulate

messages = await client.extract_messages()

table_data = [
    [msg["sender"], msg["time"], msg["message"], str(msg["attachment"])]
    for msg in messages
]
headers = ["Sender", "Time Sent", "Message", "Attachment Details"]

print("Number of previous messages: ", len(messages))
print(tabulate(table_data, headers=headers, tablefmt="grid"))

In [3]:
import asyncio
from playwright.async_api import async_playwright
import os

BASE_URL = "https://web.whatsapp.com"


async def initialize_playwright(browser_instance, user_data_dir, headless):
    # TODO: Perform browser level optimizations and other stuff
    playwright = await async_playwright().start()

    browser = await playwright[browser_instance].launch_persistent_context(
        user_data_dir, headless=headless
    )

    page_instance = await browser.new_page()

    # await page_instance.set_viewport_size({"width": 1920, "height": 1080})
    return playwright, browser, page_instance


async def login(page, user_data_dir):
    # TODO: QR code & phone number login via script

    await page.goto(BASE_URL)
    await page.bring_to_front()

    print("Waiting for WhatsApp chats to load...")
    await page.wait_for_selector(
        '//*[@id="pane-side"]/div[2]/div/div/child::div', timeout=600000
    )
    print("WhatsApp chats loaded.")

In [4]:
playwright, browser, page = await initialize_playwright("chromium", "user_data", False)
await login(page, "user_data")

Waiting for WhatsApp chats to load...
WhatsApp chats loaded.


In [38]:
async def extract_chat_details_from_side_pane(page):
    """
    Get the list of messages in the side pane (name, recent message, time, unread messages).
    """
    chat_list_div = await page.query_selector('div[aria-label="Chat list"]')
    children = await chat_list_div.query_selector_all('div[role="listitem"]')
    print(len(children))
    if children:
        print("Chat list div found.")
        all_chats = []
        for chat in children:
            name_element = await chat.query_selector('span[dir="auto"]')
            name = await name_element.inner_text() if name_element else "Unknown"

            translate_y = await chat.evaluate(
                "element => window.getComputedStyle(element).transform"
            )

            recent_message_element = await chat.query_selector(
                'div[class="_ak8k"]>span>span'
            )
            recent_message = (
                await recent_message_element.inner_text()
                if recent_message_element
                else "No recent message"
            )

            time_element = await chat.query_selector('span[dir="auto"]')
            time = (
                await time_element.inner_text() if time_element else "No time available"
            )

            unread_messages_element = await chat.query_selector(
                'span[aria-label*="unread"]'
            )
            unread_messages = (
                await unread_messages_element.inner_text()
                if unread_messages_element
                else "0"
            )

            all_chats.append(
                {
                    "name": name,
                    "recent_message": recent_message,
                    "time": time,
                    "unread_messages": unread_messages,
                    # "translate_y": float(translate_y),
                }
            )
            # all_chats.sort(key=lambda chat: chat["translate_y"])

            # print(
            #     f"Chat: {name}, Recent message: {recent_message}, Time: {time}, Unread messages: {unread_messages}"
            # )
    else:
        print("Chat list div not found.")
    return all_chats

In [39]:
await extract_chat_details_from_side_pane(page)

19
Chat list div found.


[{'name': 'SHINE FAMILY',
  'recent_message': ':\xa0',
  'time': 'SHINE FAMILY',
  'unread_messages': '138'},
 {'name': 'Asheesh',
  'recent_message': 'Photo',
  'time': 'Asheesh',
  'unread_messages': '0'},
 {'name': 'Pappa',
  'recent_message': 'Photo',
  'time': 'Pappa',
  'unread_messages': '7'},
 {'name': 'AIML Community',
  'recent_message': ':\xa0',
  'time': 'AIML Community',
  'unread_messages': '0'},
 {'name': 'GAN 2024',
  'recent_message': ':\xa0',
  'time': 'GAN 2024',
  'unread_messages': '0'},
 {'name': 'Ananth Nitte',
  'recent_message': 'Aight',
  'time': 'Ananth Nitte',
  'unread_messages': '0'},
 {'name': 'AIML Official grp 2021-25',
  'recent_message': ':\xa0',
  'time': 'AIML Official grp 2021-25',
  'unread_messages': '0'},
 {'name': 'AIML Community',
  'recent_message': ':\xa0',
  'time': 'AIML Community',
  'unread_messages': '1'},
 {'name': 'Zih',
  'recent_message': 'ohh',
  'time': 'Zih',
  'unread_messages': '0'},
 {'name': 'Prathwik',
  'recent_message': '#